# Skip-GANomaly — Pipeline B
U-Net skip-connection anomaly detector trained on normal transactions only.
**Author:** kipngeno koech (bkoech)

## 1. Dependencies

In [ ]:
# !pip install torch wandb --quiet

import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import wandb

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, classification_report,
    matthews_corrcoef,
)
from imblearn.metrics import geometric_mean_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
print(f'Device: {DEVICE}')


## 2. W&B Login

In [ ]:
# On Kaggle: Settings → Internet → ON (required for W&B)
wandb.login(key='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')  # replace with your key

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kkipngenokoech (kkipngenokoech-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Load Feature Store Data

In [ ]:
STORE = Path('/kaggle/input/notebooks/waithakafrancis/data-pipeline-v1/feature_store')

b_train = pd.read_parquet(STORE / 'pipeline_b_train.parquet')
holdout = pd.read_parquet(STORE / 'holdout.parquet')

print(f'Pipeline B train shape : {b_train.shape}')
print(f'Holdout shape          : {holdout.shape}')
print(f'Holdout fraud rate     : {holdout["Class"].mean():.4%}')
print(f'Pipeline B fraud rate  : {b_train["Class"].mean():.4%}')

assert b_train['Class'].sum() == 0, 'STOP: Pipeline B data contaminated with fraud!'
print('Assertion passed — Pipeline B is clean.')


## 4. Configuration

In [ ]:
@dataclass
class SkipGANomalyConfig:
    # ── Data ──────────────────────────────────────────────────────────────
    target_col: str         = 'Class'
    n_features: int         = 29         # V1-V28 + Amount (Time excluded per FIX 8)

    # ── Encoder / Decoder ─────────────────────────────────────────────────
    enc_dim_1: int          = 256
    enc_dim_2: int          = 128
    enc_dim_3: int          = 64
    latent_dim: int         = 32

    # ── Discriminator ─────────────────────────────────────────────────────
    disc_dim_1: int         = 128
    disc_dim_2: int         = 64

    # ── Training ──────────────────────────────────────────────────────────
    batch_size: int         = 256
    n_epochs: int           = 100
    lr: float               = 1e-4
    adam_beta1: float       = 0.5
    adam_beta2: float       = 0.999

    # ── Loss weights ──────────────────────────────────────────────────────
    w_rec: float            = 1.0    # reconstruction loss
    w_feat: float           = 1.0    # feature matching loss
    w_adv: float            = 0.001  # adversarial loss

    # ── Anomaly score weights ─────────────────────────────────────────────
    w_score_rec: float      = 0.9
    w_score_feat: float     = 0.1


# Time excluded per FIX 8
MODEL_FEATURES = [f'V{i}' for i in range(1, 29)] + ['Amount']  # 29 features
FEATURE_COLS   = MODEL_FEATURES

cfg            = SkipGANomalyConfig()
cfg.n_features = len(MODEL_FEATURES)   # 29

print(f'Config: n_features={cfg.n_features}, latent_dim={cfg.latent_dim}')


## 5. Initialise W&B Run

In [ ]:
run = wandb.init(
    project = 'PRINCIPLES AND ENGINEERING  APPLICATIONS OF AI',
    name    = 'kip-skip-ganomaly-pipeline-2',
    tags    = ['Skip-GANomaly', 'Pipeline-B', 'anomaly-detection'],
    config  = asdict(cfg),   # every field in SkipGANomalyConfig logged
)

# Sync config back for sweep compatibility
wcfg = wandb.config
cfg.enc_dim_1     = wcfg.enc_dim_1
cfg.enc_dim_2     = wcfg.enc_dim_2
cfg.enc_dim_3     = wcfg.enc_dim_3
cfg.latent_dim    = wcfg.latent_dim
cfg.disc_dim_1    = wcfg.disc_dim_1
cfg.disc_dim_2    = wcfg.disc_dim_2
cfg.batch_size    = wcfg.batch_size
cfg.n_epochs      = wcfg.n_epochs
cfg.lr            = wcfg.lr
cfg.w_rec         = wcfg.w_rec
cfg.w_feat        = wcfg.w_feat
cfg.w_adv         = wcfg.w_adv
cfg.w_score_rec   = wcfg.w_score_rec
cfg.w_score_feat  = wcfg.w_score_feat

print('W&B run initialised:', run.name)
print('W&B project URL    :', run.url)


wandb: setting up run md3rmn1y
wandb: Tracking run with wandb version 0.24.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260308_210301-md3rmn1y
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run kip-skip-ganomaly-pipeline-2
wandb: ⭐️ View project at https://wandb.ai/kkipngenokoech-carnegie-mellon-university/PRINCIPLES%20AND%20ENGINEERING%20%20APPLICATIONS%20OF%20AI
wandb: 🚀 View run at https://wandb.ai/kkipngenokoech-carnegie-mellon-university/PRINCIPLES%20AND%20ENGINEERING%20%20APPLICATIONS%20OF%20AI/runs/md3rmn1y


W&B run initialised: kip-skip-ganomaly-pipeline-2
W&B project URL    : https://wandb.ai/kkipngenokoech-carnegie-mellon-university/PRINCIPLES%20AND%20ENGINEERING%20%20APPLICATIONS%20OF%20AI/runs/md3rmn1y


## 6. Data Preparation

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Internal val split from b_train (normal only) — consistent with GANomaly
X_all = b_train[FEATURE_COLS].values.astype(np.float32)
X_train_raw, X_val_raw = train_test_split(X_all, test_size=0.2, random_state=GLOBAL_SEED)

sgan_scaler = StandardScaler()
X_train_np  = np.clip(sgan_scaler.fit_transform(X_train_raw), -5, 5).astype(np.float32)
X_val_np    = np.clip(sgan_scaler.transform(X_val_raw),       -5, 5).astype(np.float32)

# Full holdout (natural distribution — both normal and fraud)
X_holdout_np = np.clip(
    sgan_scaler.transform(holdout[FEATURE_COLS].values.astype(np.float32)), -5, 5
).astype(np.float32)
y_holdout_np = holdout[cfg.target_col].values

print(f'Train (normal only)      : {X_train_np.shape}')
print(f'Val   (normal only)      : {X_val_np.shape}')
print(f'Holdout (full, natural)  : {X_holdout_np.shape} | '
      f'Fraud: {y_holdout_np.sum()} ({y_holdout_np.mean():.4%})')
print(f'Scaled range             : {X_train_np.min():.3f} to {X_train_np.max():.3f}')

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_np, dtype=torch.float32)),
    batch_size=cfg.batch_size, shuffle=True, drop_last=True
)
print(f'Batches per epoch: {len(train_loader)}')

wandb.log({
    'data/n_train_normal'    : len(X_train_np),
    'data/n_val_normal'      : len(X_val_np),
    'data/n_holdout'         : len(X_holdout_np),
    'data/holdout_fraud_rate': float(y_holdout_np.mean()),
    'data/batches_per_epoch' : len(train_loader),
})


## 7. Skip-GANomaly Architecture

In [ ]:
class Encoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        dims = [cfg.n_features, cfg.enc_dim_1, cfg.enc_dim_2, cfg.enc_dim_3, cfg.latent_dim]
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dims[i], dims[i+1]),
                nn.BatchNorm1d(dims[i+1]),
                nn.LeakyReLU(0.2, inplace=True)
            ) for i in range(len(dims) - 1)
        ])

    def forward(self, x):
        skip_outputs, h = [], x
        for i, layer in enumerate(self.layers):
            h = layer(h)
            if i < len(self.layers) - 1:
                skip_outputs.append(h)
        return h, skip_outputs


class Decoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        dims = [cfg.latent_dim, cfg.enc_dim_3, cfg.enc_dim_2, cfg.enc_dim_1, cfg.n_features]
        self.layers = nn.ModuleList()
        for i in range(len(dims) - 1):
            is_last = (i == len(dims) - 2)
            self.layers.append(
                nn.Linear(dims[i], dims[i+1]) if is_last else
                nn.Sequential(
                    nn.Linear(dims[i], dims[i+1]),
                    nn.BatchNorm1d(dims[i+1]),
                    nn.ReLU(inplace=True)
                )
            )

    def forward(self, z, skip_outputs):
        h = z
        skips = list(reversed(skip_outputs))
        for i, layer in enumerate(self.layers):
            h = layer(h)
            if i < len(skips) and h.shape == skips[i].shape:
                h = h + skips[i]
        return h


class Discriminator(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(cfg.n_features, cfg.disc_dim_1), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(cfg.disc_dim_1, cfg.disc_dim_2), nn.LeakyReLU(0.2, inplace=True),
        )
        self.classifier = nn.Linear(cfg.disc_dim_2, 1)

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features), features


encoder = Encoder(cfg).to(DEVICE)
decoder = Decoder(cfg).to(DEVICE)
disc    = Discriminator(cfg).to(DEVICE)

n_enc  = sum(p.numel() for p in encoder.parameters())
n_dec  = sum(p.numel() for p in decoder.parameters())
n_disc = sum(p.numel() for p in disc.parameters())
print(f'Encoder params     : {n_enc:,}')
print(f'Decoder params     : {n_dec:,}')
print(f'Discriminator params: {n_disc:,}')

# Smoke test
dummy = torch.randn(4, cfg.n_features).to(DEVICE)
z_d, sk_d = encoder(dummy)
r_d = decoder(z_d, sk_d)
print(f'Forward pass OK — input: {dummy.shape} | latent: {z_d.shape} | recon: {r_d.shape}')

wandb.log({
    'model/encoder_params'      : n_enc,
    'model/decoder_params'      : n_dec,
    'model/discriminator_params': n_disc,
    'model/total_params'        : n_enc + n_dec + n_disc,
})


Encoder params     : 57,248
Decoder params     : 57,202
Discriminator params: 14,849
Forward pass OK — input: torch.Size([4, 50]) | latent: torch.Size([4, 32]) | recon: torch.Size([4, 50])


## 8. Training Loop

In [ ]:
opt_gen  = optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=cfg.lr, betas=(cfg.adam_beta1, cfg.adam_beta2)
)
opt_disc = optim.Adam(disc.parameters(), lr=cfg.lr, betas=(cfg.adam_beta1, cfg.adam_beta2))

criterion_adv  = nn.BCEWithLogitsLoss()
criterion_rec  = nn.MSELoss()
criterion_feat = nn.MSELoss()


def train_skip_ganomaly(encoder, decoder, disc, loader, cfg, device):
    history = {'g_loss': [], 'd_loss': [], 'rec_loss': [], 'feat_loss': []}

    for epoch in range(cfg.n_epochs):
        encoder.train(); decoder.train(); disc.train()
        g_losses, d_losses, rec_losses, feat_losses = [], [], [], []

        for (x_real,) in loader:
            x_real = x_real.to(device)
            bs = x_real.size(0)
            real_labels = torch.ones(bs,  1, device=device)
            fake_labels = torch.zeros(bs, 1, device=device)

            z, skips = encoder(x_real)
            x_recon  = decoder(z, skips)

            # Discriminator
            logit_real, _  = disc(x_real)
            logit_fake, _  = disc(x_recon.detach())
            d_loss = 0.5 * (
                criterion_adv(logit_real, real_labels) +
                criterion_adv(logit_fake, fake_labels)
            )
            opt_disc.zero_grad(); d_loss.backward(); opt_disc.step()

            # Generator
            logit_fake_g, feat_fake = disc(x_recon)
            _,            feat_real = disc(x_real)
            loss_rec  = criterion_rec(x_recon, x_real)
            loss_feat = criterion_feat(feat_fake, feat_real.detach())
            loss_adv  = criterion_adv(logit_fake_g, real_labels)
            g_loss = cfg.w_rec * loss_rec + cfg.w_feat * loss_feat + cfg.w_adv * loss_adv
            opt_gen.zero_grad(); g_loss.backward(); opt_gen.step()

            g_losses.append(g_loss.item()); d_losses.append(d_loss.item())
            rec_losses.append(loss_rec.item()); feat_losses.append(loss_feat.item())

        mg = np.mean(g_losses); md_ = np.mean(d_losses)
        mr = np.mean(rec_losses); mf = np.mean(feat_losses)
        history['g_loss'].append(mg); history['d_loss'].append(md_)
        history['rec_loss'].append(mr); history['feat_loss'].append(mf)

        # ── W&B: log every epoch ──────────────────────────────────────────
        wandb.log({
            'train/epoch'                    : epoch + 1,
            'train/g_loss'                   : mg,
            'train/d_loss'                   : md_,
            'train/reconstruction_loss'      : mr,
            'train/feature_matching_loss'    : mf,
            'train/rec_to_feat_ratio'        : mr / (mf + 1e-8),
            'train/weighted_rec_component'   : cfg.w_rec  * mr,
            'train/weighted_feat_component'  : cfg.w_feat * mf,
            'train/weighted_adv_component'   : cfg.w_adv  * np.mean([criterion_adv(
                torch.zeros(1,1), torch.zeros(1,1)).item()]),  # placeholder shape
        }, step=epoch + 1)

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1:3d}/{cfg.n_epochs}] G: {mg:.4f} | D: {md_:.4f} | Rec: {mr:.4f} | Feat: {mf:.4f}')

    return history


print('Starting Skip-GANomaly training...')
t0 = time.time()
history = train_skip_ganomaly(encoder, decoder, disc, train_loader, cfg, DEVICE)
train_time_min = (time.time() - t0) / 60
print(f'Training complete in {train_time_min:.1f} min')

wandb.log({'train/total_time_min': train_time_min})


Starting Skip-GANomaly training...


wandb: WARNING Tried to log to step 1 that is less than the current step 2. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


Epoch [ 10/100] G: 0.0085 | D: 0.6925 | Rec: 0.0074 | Feat: 0.0004
Epoch [ 20/100] G: 0.0064 | D: 0.6926 | Rec: 0.0054 | Feat: 0.0003
Epoch [ 30/100] G: 0.0055 | D: 0.6927 | Rec: 0.0046 | Feat: 0.0003
Epoch [ 40/100] G: 0.0051 | D: 0.6925 | Rec: 0.0041 | Feat: 0.0003
Epoch [ 50/100] G: 0.0049 | D: 0.6901 | Rec: 0.0038 | Feat: 0.0004
Epoch [ 60/100] G: 0.0059 | D: 0.6762 | Rec: 0.0037 | Feat: 0.0015
Epoch [ 70/100] G: 0.0076 | D: 0.6528 | Rec: 0.0037 | Feat: 0.0031
Epoch [ 80/100] G: 0.0097 | D: 0.6236 | Rec: 0.0037 | Feat: 0.0052
Epoch [ 90/100] G: 0.0120 | D: 0.5930 | Rec: 0.0038 | Feat: 0.0073
Epoch [100/100] G: 0.0139 | D: 0.5670 | Rec: 0.0039 | Feat: 0.0091
Training complete in 21.7 min


## 9. Anomaly Scoring

In [ ]:
def compute_anomaly_scores(encoder, decoder, disc, X, cfg, device, batch_size=512):
    encoder.eval(); decoder.eval(); disc.eval()
    all_scores = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.tensor(X[start:start+batch_size], dtype=torch.float32).to(device)
            z, skips      = encoder(xb)
            x_recon       = decoder(z, skips)
            _, feat_real  = disc(xb)
            _, feat_recon = disc(x_recon)
            rec_err  = ((xb - x_recon) ** 2).mean(dim=1)
            feat_err = ((feat_real - feat_recon) ** 2).mean(dim=1)
            all_scores.append((cfg.w_score_rec * rec_err + cfg.w_score_feat * feat_err).cpu().numpy())
    return np.concatenate(all_scores)


print('Computing anomaly scores on validation set...')
val_scores = compute_anomaly_scores(encoder, decoder, disc, X_val_np, cfg, DEVICE)

wandb.log({
    'scoring/val_score_min'  : float(val_scores.min()),
    'scoring/val_score_max'  : float(val_scores.max()),
    'scoring/val_score_mean' : float(val_scores.mean()),
    'scoring/val_score_std'  : float(val_scores.std()),
})
print(f'Val scores — min: {val_scores.min():.4f} | max: {val_scores.max():.4f} | mean: {val_scores.mean():.4f}')

# Inversion check — reconstruction-based detectors sometimes rank inverted
_y_val_binary = (y_val_np > 0).astype(int) if y_val_np.max() > 1 else y_val_np
_raw_auroc = roc_auc_score(_y_val_binary, val_scores) if len(np.unique(_y_val_binary)) > 1 else 0.5
print(f'Raw AUROC on val set: {_raw_auroc:.4f}')
if _raw_auroc < 0.5:
    print('INVERTED — flipping anomaly scores.')
    val_scores = -val_scores
    print(f'Corrected AUROC: {1 - _raw_auroc:.4f}')
else:
    print('Scores orientation correct.')

Computing anomaly scores on validation set...
Val scores — min: 0.0001 | max: 0.3241 | mean: 0.0007
Raw AUROC on val set: 0.9434
Scores orientation correct.


## 10. Threshold Search on Validation Set

In [ ]:
score_scaler    = MinMaxScaler()
val_scores_norm = score_scaler.fit_transform(val_scores.reshape(-1, 1)).ravel()

# Threshold = 95th percentile of normal val scores (leakage-free, FIX 3)
best_thresh = float(np.percentile(val_scores_norm, 95))

print(f'Threshold (val 95th percentile): {best_thresh:.4f}')
print(f'Val false-positive rate at this threshold: {(val_scores_norm >= best_thresh).mean():.4%}')

wandb.log({
    'eval/val_score_mean'  : float(val_scores_norm.mean()),
    'eval/val_score_std'   : float(val_scores_norm.std()),
    'eval/val_score_p95'   : float(np.percentile(val_scores_norm, 95)),
    'eval/val_score_p99'   : float(np.percentile(val_scores_norm, 99)),
    'eval/threshold_source': 'val_95th_percentile',
    'eval/threshold_value' : best_thresh,
})


## 11. Evaluation on Fraud Holdout

In [ ]:
# ── Holdout evaluation at natural class distribution ──────────────────────
# Threshold from val normal 95th percentile (FIX 3).
# Evaluated once on full holdout at natural fraud rate (FIX 4).

t_infer           = time.time()
holdout_scores    = compute_anomaly_scores(encoder, decoder, disc, X_holdout_np, cfg, DEVICE)
inference_time_ms = (time.time() - t_infer) * 1000

holdout_scores_norm = score_scaler.transform(holdout_scores.reshape(-1, 1)).ravel()

# Inversion check using holdout labels
_raw_auroc = roc_auc_score(y_holdout_np, holdout_scores_norm)
if _raw_auroc < 0.5:
    print('INVERTED — flipping holdout scores.')
    holdout_scores_norm = -holdout_scores_norm

holdout_preds = (holdout_scores_norm >= best_thresh).astype(int)

f1        = f1_score(y_holdout_np,        holdout_preds, zero_division=0)
precision = precision_score(y_holdout_np, holdout_preds, zero_division=0)
recall    = recall_score(y_holdout_np,    holdout_preds, zero_division=0)
auroc     = roc_auc_score(y_holdout_np,   holdout_scores_norm)
auprc     = average_precision_score(y_holdout_np, holdout_scores_norm)
mcc       = matthews_corrcoef(y_holdout_np, holdout_preds)
gmean     = geometric_mean_score(y_holdout_np, holdout_preds)

print(f'\nHoldout set: {len(y_holdout_np):,} records | '
      f'Fraud: {y_holdout_np.sum()} ({y_holdout_np.mean():.4%}) | '
      f'Normal: {(y_holdout_np == 0).sum():,}')
print('Natural class distribution — no artificial fraud rate inflation (FIX 4).')

print('\n' + '=' * 55)
print('  Skip-GANomaly — Holdout Evaluation Results')
print('=' * 55)
print(f'  F1-Score       : {f1:.4f}')
print(f'  Precision      : {precision:.4f}')
print(f'  Recall         : {recall:.4f}')
print(f'  AUROC          : {auroc:.4f}')
print(f'  AUPRC          : {auprc:.4f}')
print(f'  MCC            : {mcc:.4f}')
print(f'  G-mean         : {gmean:.4f}')
print(f'  Threshold      : {best_thresh:.4f}  (val 95th percentile)')
print(f'  Inference time : {inference_time_ms:.2f} ms ({len(X_holdout_np):,} records)')
print('=' * 55)
print(classification_report(y_holdout_np, holdout_preds, target_names=['Legit', 'Fraud']))

wandb.log({
    'holdout/f1'               : f1,
    'holdout/precision'        : precision,
    'holdout/recall'           : recall,
    'holdout/auroc'            : auroc,
    'holdout/auprc'            : auprc,
    'holdout/mcc'              : mcc,
    'holdout/gmean'            : gmean,
    'holdout/threshold'        : best_thresh,
    'holdout/n_records'        : len(y_holdout_np),
    'holdout/fraud_rate'       : float(y_holdout_np.mean()),
    'holdout/inference_time_ms': inference_time_ms,
})
wandb.run.summary.update({
    'best_f1': f1, 'best_auroc': auroc, 'best_auprc': auprc,
    'best_recall': recall, 'best_precision': precision,
    'best_mcc': mcc, 'best_gmean': gmean,
})


## 12. Holdout-Only Fraud Recall

In [ ]:
# Cell superseded — pure-fraud-only holdout no longer used.
# Holdout evaluation with MCC/G-mean is in the cell above (natural distribution).


## 13. Save Results & Finish W&B Run

In [ ]:
results = {
    'model'             : 'Skip-GANomaly',
    'pipeline'          : 'B',
    'author'            : 'kipngeno koech (bkoech)',
    'f1'                : round(f1, 4),
    'precision'         : round(precision, 4),
    'recall'            : round(recall, 4),
    'auroc'             : round(auroc, 4),
    'auprc'             : round(auprc, 4),
    'mcc'               : round(mcc, 4),
    'gmean'             : round(gmean, 4),
    'threshold'         : round(best_thresh, 4),
    'architecture': {
        'enc_dims'      : [cfg.enc_dim_1, cfg.enc_dim_2, cfg.enc_dim_3],
        'latent_dim'    : cfg.latent_dim,
        'disc_dims'     : [cfg.disc_dim_1, cfg.disc_dim_2],
        'n_features'    : cfg.n_features,
        'w_rec'         : cfg.w_rec,
        'w_feat'        : cfg.w_feat,
        'w_adv'         : cfg.w_adv,
        'w_score_rec'   : cfg.w_score_rec,
        'w_score_feat'  : cfg.w_score_feat,
    },
    'wandb_run_url': wandb.run.url,
}

out_path = Path('/kaggle/working/skip_ganomaly_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

wandb.save(str(out_path))
print(f'Results saved to {out_path}')
print(json.dumps(results, indent=2))

wandb.finish()
print('W&B run finished.')
